# Goodgorithm — Category classifier training

Trains the classifier that replaces `processing/src/pipeline_stages/taxonomy.py` as the primary category-assignment mechanism — the keyword matcher's exact-match recall ceiling meant most categories showed well under 10 posts at any time. See `CLAUDE.md`'s Category filtering section for full context.

**What this notebook does:** loads `cardiffnlp/tweet_topic_multi` (real tweets, 19 topic labels), remaps a trimmed 4-label subset onto Goodgorithm's category taxonomy, trains a TF-IDF + one-vs-rest logistic regression classifier, evaluates it two ways (per-label multi-label metrics, and a simulated deployed-policy confusion matrix against the *full* test set including off-taxonomy posts — see the Evaluate section for why this second check matters), exports to ONNX, and uploads it to the same `goodgorithm-models` R2 bucket the sentiment model uses, under a `category-classifier/` prefix.

**Unlike the sentiment notebook, this doesn't need a GPU** — TF-IDF + logistic regression trains on CPU in a couple of minutes. Kept as a notebook anyway for parity with the sentiment model's audit trail and because HuggingFace `datasets` download + a human eyeballing results before publishing still benefits from the same manual ritual.

**Before running:** this notebook fetches `processing/src/util/text_normalize.py` from a pinned commit (not a live fetch from `main`), the same reason the sentiment notebook pins `sentiment_model.py` — so normalization can never drift between training and inference even though they run in different environments. If that file changes, update `TEXT_NORMALIZE_COMMIT` below.

In [ ]:
!pip install -q datasets scikit-learn skl2onnx onnxruntime boto3

## Fetch the shared text-normalization file

In [ ]:
TEXT_NORMALIZE_COMMIT = "c8932626c7cfab2a60edd6d0d8d1c37dd87d4beb"

import urllib.request

url = (
    f"https://raw.githubusercontent.com/goodgorithm/goodgorithm/"
    f"{TEXT_NORMALIZE_COMMIT}/processing/src/util/text_normalize.py"
)
urllib.request.urlretrieve(url, "text_normalize.py")

import text_normalize

print("fetched text_normalize.py @", TEXT_NORMALIZE_COMMIT)

## Load the dataset

`cardiffnlp/tweet_topic_multi` ships several split "families" (`_2020`/`_2021`/`_all`/`_random`/`_coling2022*`). We train on `train_2020`/`validation_2020` and hold out `test_2021` untouched as an independent, temporally-shifted eval — the same reasoning the sentiment notebook uses for TweetEval's own reserved test split: it's the number that best predicts real-world performance, since nothing from it ever touched training.

In [ ]:
from datasets import load_dataset

ds = load_dataset("cardiffnlp/tweet_topic_multi")
for split in ["train_2020", "validation_2020", "test_2021"]:
    print(split, len(ds[split]))

## Remap to Goodgorithm's 4 categories

The original dataset has 19 labels. We keep 4 (`arts_culture`, `food_dining`, `gaming`, `science_technology`) — narrowed from an earlier 8-category set (issue #37) once real production precision data showed a sharp, specific divide: these four measured 60-95% real sampled precision in production, the other four (`sports`, `health_fitness`, `learning_education`, `travel_adventure`) measured 23-52%, each with an identifiable, repeatable confusion pattern (e.g. `sports` picking up general "game"-keyword bleed from video-game posts and political news). See `CLAUDE.md`'s Category filtering section for the full numbers and reasoning. `arts_culture` absorbs three original labels (`arts_&_culture`, `film_tv_&_video`, `music`) to maximize its volume, same as before this narrowing.

**Fixed, alphabetical output order** — this becomes the classifier's output column order, recorded verbatim in `config.json` and asserted against at inference load time. Getting this wrong is a silent-wrong-answer failure (the model loads fine, just mislabels everything), not a crash — hence the explicit assertion later in this notebook.

In [ ]:
ORIGINAL_LABELS = [
    "arts_&_culture", "business_&_entrepreneurs", "celebrity_&_pop_culture",
    "diaries_&_daily_life", "family", "fashion_&_style", "film_tv_&_video",
    "fitness_&_health", "food_&_dining", "gaming", "learning_&_educational",
    "music", "news_&_social_concern", "other_hobbies", "relationships",
    "science_&_technology", "sports", "travel_&_adventure", "youth_&_student_life",
]

LABEL_MAP = {
    "arts_culture": ["arts_&_culture", "film_tv_&_video", "music"],
    "food_dining": ["food_&_dining"],
    "gaming": ["gaming"],
    "science_technology": ["science_&_technology"],
}
KEPT_LABELS = sorted(LABEL_MAP.keys())
orig_index = {name: i for i, name in enumerate(ORIGINAL_LABELS)}

import numpy as np


def remap(multi_hot):
    out = np.zeros(len(KEPT_LABELS), dtype=int)
    for j, cat in enumerate(KEPT_LABELS):
        if any(multi_hot[orig_index[src]] for src in LABEL_MAP[cat]):
            out[j] = 1
    return out


def load_split(split_name):
    """Kept-label-only examples - for training and standard multi-label eval."""
    texts, labels = [], []
    for ex in ds[split_name]:
        remapped = remap(ex["label"])
        if remapped.sum() == 0:
            continue
        texts.append(ex["text"])
        labels.append(remapped)
    return texts, np.array(labels)


def load_split_full(split_name):
    """Every example, including ones whose only original labels are
    outside our kept 4 (news_&_social_concern, diaries_&_daily_life,
    sports, fitness_&_health, learning_&_educational, travel_&_adventure,
    etc.) - these should ideally get category=None. Needed later to test
    abstain behavior honestly: load_split()'s pre-filtering would
    otherwise hide the model's single most important failure mode by
    construction."""
    texts, golds = [], []
    for ex in ds[split_name]:
        remapped = remap(ex["label"])
        idxs = np.where(remapped == 1)[0]
        texts.append(ex["text"])
        golds.append(KEPT_LABELS[idxs[0]] if len(idxs) else None)
    return texts, golds


train_texts, train_labels = load_split("train_2020")
val_texts, val_labels = load_split("validation_2020")
test_texts, test_labels = load_split("test_2021")

print(f"train_2020: {len(ds['train_2020'])} -> {len(train_texts)} kept")
print(f"validation_2020: {len(ds['validation_2020'])} -> {len(val_texts)} kept")
print(f"test_2021: {len(ds['test_2021'])} -> {len(test_texts)} kept")
print("\nPer-label positive counts (train) - a class with only a few hundred examples is a real yellow flag:")
for j, cat in enumerate(KEPT_LABELS):
    print(f"  {cat}: {int(train_labels[:, j].sum())}")

## Vectorize + train

`TfidfVectorizer` here is fit **once** on the training corpus and frozen — structurally different from `processing/src/pipeline_stages/topicality.py`'s same-named class, which refits per-batch for within-batch relative rarity. This one is versioned as a model artifact, the TF-IDF analog of the sentiment CNN's `vocab.json`. Easy point of confusion given the superficial code similarity, worth flagging explicitly.

Trained genuinely multi-label (matching the dataset's native annotations, not collapsed to single-label at training time — that would discard real signal). Deployed storage stays single-label (top-1-above-threshold); multi-label storage/API/UI is a legitimate fast-follow, out of scope here.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.multiclass import OneVsRestClassifier

train_norm = [text_normalize.normalize_text(t) for t in train_texts]
val_norm = [text_normalize.normalize_text(t) for t in val_texts]
test_norm = [text_normalize.normalize_text(t) for t in test_texts]

vectorizer = TfidfVectorizer(
    ngram_range=(1, 2), stop_words="english", sublinear_tf=True, min_df=2
)
X_train = vectorizer.fit_transform(train_norm)
X_test = vectorizer.transform(test_norm)
print("vocab size:", len(vectorizer.vocabulary_))

clf = OneVsRestClassifier(LogisticRegression(class_weight="balanced", max_iter=1000))
clf.fit(X_train, train_labels)
print("trained.")

## Evaluate

Two checks, both needed because training data is multi-label but production storage is single-label:

1. **Multi-label metrics** on `test_2021`'s kept-label-only examples — per-label precision/recall/F1 against the multi-hot ground truth.
2. **Simulated deployed-policy confusion matrix**, on the *full* `test_2021` set — including posts whose only original labels were dropped (`news_&_social_concern`, `diaries_&_daily_life`, etc.), which should ideally get `category=None`. Using only kept-label-having examples here (like check 1 does) would hide the model's single most important failure mode — never abstaining on genuinely off-taxonomy content — by construction. This second number is what actually predicts what a production diagnostic query (grouping `processed_posts` by `category`) will show.

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix

test_probs = clf.predict_proba(X_test)
test_pred_binary = (test_probs >= 0.5).astype(int)
print(classification_report(test_labels, test_pred_binary, target_names=KEPT_LABELS, zero_division=0))

In [ ]:
full_test_texts, full_test_golds = load_split_full("test_2021")
full_test_norm = [text_normalize.normalize_text(t) for t in full_test_texts]
full_test_probs = clf.predict_proba(vectorizer.transform(full_test_norm))

n_off_taxonomy = sum(1 for g in full_test_golds if g is None)
print(f"full test_2021: {len(full_test_texts)} examples, {n_off_taxonomy} off-taxonomy (gold=None)")


def deployed_predict(probs, threshold):
    preds = []
    for row in probs:
        best_idx = int(np.argmax(row))
        preds.append(KEPT_LABELS[best_idx] if row[best_idx] >= threshold else None)
    return preds


print("\nThreshold sweep (accuracy vs single-label gold, off-taxonomy correctly-abstained rate):")
sweep_results = []
for threshold in [0.35, 0.45, 0.55, 0.65, 0.75, 0.85]:
    preds = deployed_predict(full_test_probs, threshold)
    correct = sum(1 for g, p in zip(full_test_golds, preds) if g == p)
    accuracy = correct / len(full_test_golds)
    off_correct = sum(1 for g, p in zip(full_test_golds, preds) if g is None and p is None)
    off_rate = off_correct / n_off_taxonomy if n_off_taxonomy else 0.0
    none_rate = sum(1 for p in preds if p is None) / len(preds)
    sweep_results.append((threshold, accuracy, off_rate, none_rate))
    print(f"  threshold={threshold:.2f}: accuracy={accuracy:.3f} off_taxonomy_correctly_abstained={off_rate:.3f} overall_none_rate={none_rate:.3f}")

**Threshold choice**: the actual goal is *more* categorized volume, not maximum abstain-precision — a threshold that trades away a lot of on-taxonomy accuracy for marginally better off-topic rejection works against the point of this whole change. Pick the knee of the sweep curve above — hand-picked from real numbers each run produces, not a mechanical formula, and not necessarily the same value run to run (the 8-category notebook's knee was typically ~0.55; after narrowing to 4 categories, issue #37, the knee moved to ~0.65 — fewer competing classes shifted the softmax distribution shape, so don't assume the old value carries over). **Re-check this against the printed sweep before publishing** — the exact numbers will vary run to run.

In [ ]:
CONFIDENCE_THRESHOLD = 0.65  # confirm against the sweep printed above before publishing

NONE = "(none)"
labels_for_cm = KEPT_LABELS + [NONE]
cm_golds = [g if g is not None else NONE for g in full_test_golds]
deployed_preds = deployed_predict(full_test_probs, CONFIDENCE_THRESHOLD)
cm_preds = [p if p is not None else NONE for p in deployed_preds]
cm = confusion_matrix(cm_golds, cm_preds, labels=labels_for_cm)
print("confusion matrix (rows=gold, cols=predicted):", labels_for_cm)
print(cm)

correct = sum(1 for g, p in zip(full_test_golds, deployed_preds) if g == p)
accuracy = correct / len(full_test_golds)
off_correct = sum(1 for g, p in zip(full_test_golds, deployed_preds) if g is None and p is None)
off_rate = off_correct / n_off_taxonomy if n_off_taxonomy else 0.0
print(f"\ndeployed-policy accuracy: {accuracy:.3f}")
print(f"off-taxonomy correctly abstained: {off_rate:.3f}")

## Spot-check

Hand-written examples — a model acing the metrics above but failing obvious spot-checks is a red flag worth catching before publishing. Replace/extend with real production examples pulled from staging when available. Includes a couple of deliberately off-taxonomy examples (including the exact "game"-keyword sports/gaming bleed that was part of why `sports` got dropped, issue #37) to confirm the model abstains on them rather than leaking into a kept category.

In [ ]:
spot_checks = [
    "Just adopted the sweetest rescue puppy, she's already best friends with the cat!",  # off-taxonomy (animals) -- expect abstain
    "New indie album dropped today and it's on repeat already",  # arts_culture
    "Finally beat the final boss after 40 hours, what a game",  # gaming
    "Our local team just won the championship, what a game",  # off-taxonomy (sports, dropped) -- the real game-keyword bleed pattern, expect abstain, watch for gaming leakage
    "Tried a new ramen spot downtown and it was incredible",  # food_dining
    "Scientists just published a breakthrough paper on quantum computing",  # science_technology
    "Just booked flights for a backpacking trip through the Alps",  # off-taxonomy (travel, dropped) -- expect abstain
]
for text in spot_checks:
    vec = vectorizer.transform([text_normalize.normalize_text(text)])
    probs = clf.predict_proba(vec)[0]
    best_idx = int(np.argmax(probs))
    pred = KEPT_LABELS[best_idx] if probs[best_idx] >= CONFIDENCE_THRESHOLD else None
    print(f"[{pred}] ({probs[best_idx]:.2f}) {text}")

## Export to ONNX

Verify numerically against `pipeline.predict_proba()`, and explicitly assert the exported graph's output column order matches `KEPT_LABELS` — the label-order-mismatch failure mode is silent (model loads fine, confidently mislabels everything), not a crash.

In [ ]:
from skl2onnx import convert_sklearn
from skl2onnx.common.data_types import StringTensorType
from sklearn.pipeline import Pipeline
import onnxruntime as ort

pipeline = Pipeline([("tfidf", vectorizer), ("clf", clf)])
onnx_model = convert_sklearn(
    pipeline,
    initial_types=[("input", StringTensorType([None, 1]))],
    options={id(clf): {"zipmap": False}},
)
onnx_bytes = onnx_model.SerializeToString()

sess = ort.InferenceSession(onnx_bytes, providers=["CPUExecutionProvider"])
sample = np.array([[text_normalize.normalize_text(t)] for t in spot_checks[:3]], dtype=object)
onnx_out = sess.run(None, {"input": sample})
output_names = [o.name for o in sess.get_outputs()]
assert output_names[1] == "probabilities", f"unexpected ONNX output order: {output_names}"
onnx_probs = onnx_out[1]
sklearn_probs = clf.predict_proba(vectorizer.transform([text_normalize.normalize_text(t) for t in spot_checks[:3]]))
assert np.allclose(onnx_probs, sklearn_probs, atol=1e-3), "ONNX/sklearn parity check failed"
assert onnx_probs.shape[1] == len(KEPT_LABELS), "ONNX output width doesn't match KEPT_LABELS"
print("ONNX export verified: parity OK, output width matches", len(KEPT_LABELS), "labels")

with open("model.onnx", "wb") as f:
    f.write(onnx_bytes)

## Package `config.json`

In [ ]:
import json
from datetime import datetime, timezone

VERSION = "v3"

config = {
    "version": VERSION,
    "text_normalize_source_commit": TEXT_NORMALIZE_COMMIT,
    "labels": KEPT_LABELS,
    "confidence_threshold": CONFIDENCE_THRESHOLD,
    "tfidf": {"ngram_range": [1, 2], "stop_words": "english", "sublinear_tf": True, "min_df": 2},
    "dataset_composition": {
        "source": "cardiffnlp/tweet_topic_multi",
        "train_split": "train_2020",
        "validation_split": "validation_2020",
        "test_split": "test_2021",
        "train_examples_kept": len(train_texts),
        "validation_examples_kept": len(val_texts),
        "test_examples_kept": len(test_texts),
    },
    "per_label_train_support": {cat: int(train_labels[:, j].sum()) for j, cat in enumerate(KEPT_LABELS)},
    "threshold_sweep": [
        {"threshold": t, "accuracy": a, "off_taxonomy_correctly_abstained": o, "overall_none_rate": n}
        for t, a, o, n in sweep_results
    ],
    "deployed_policy_accuracy_test2021": accuracy,
    "off_taxonomy_correctly_abstained_test2021": off_rate,
    "trained_at": datetime.now(timezone.utc).isoformat(),
}

with open("config.json", "w") as f:
    json.dump(config, f, indent=2)

print(json.dumps(config, indent=2))

## Upload to R2

Uploads always happen unconditionally — versioned artifacts are cheap and safe to publish; promoting a version to production is a separate, deliberate step (see below). Only two artifacts here, not three like the sentiment model — there's no separate `vocab.json`-equivalent, the TF-IDF vocabulary is baked into the exported ONNX model.

In [ ]:
R2_ACCOUNT_ID = R2_ACCESS_KEY_ID = R2_SECRET_ACCESS_KEY = R2_BUCKET_NAME = None

try:
    from google.colab import userdata

    R2_ACCOUNT_ID = userdata.get("R2_ACCOUNT_ID")
    R2_ACCESS_KEY_ID = userdata.get("R2_ACCESS_KEY_ID")
    R2_SECRET_ACCESS_KEY = userdata.get("R2_SECRET_ACCESS_KEY")
    R2_BUCKET_NAME = userdata.get("R2_BUCKET_NAME")
except Exception as e:
    print(f"Colab secrets unavailable ({type(e).__name__}: {e}), trying Kaggle secrets...")

if not R2_ACCOUNT_ID:
    try:
        from kaggle_secrets import UserSecretsClient

        secrets = UserSecretsClient()
        R2_ACCOUNT_ID = secrets.get_secret("R2_ACCOUNT_ID")
        R2_ACCESS_KEY_ID = secrets.get_secret("R2_ACCESS_KEY_ID")
        R2_SECRET_ACCESS_KEY = secrets.get_secret("R2_SECRET_ACCESS_KEY")
        R2_BUCKET_NAME = secrets.get_secret("R2_BUCKET_NAME")
    except Exception as e:
        print(f"Kaggle secrets unavailable ({type(e).__name__}: {e}), falling back to manual values...")

if not R2_ACCOUNT_ID:
    # Manual fallback -- fill these in locally, never commit real values.
    R2_ACCOUNT_ID = ""
    R2_ACCESS_KEY_ID = ""
    R2_SECRET_ACCESS_KEY = ""
    R2_BUCKET_NAME = ""

assert R2_ACCOUNT_ID and R2_ACCESS_KEY_ID and R2_SECRET_ACCESS_KEY and R2_BUCKET_NAME, (
    "R2 credentials not set -- see the markdown cell above"
)

In [ ]:
import boto3

s3 = boto3.client(
    "s3",
    endpoint_url=f"https://{R2_ACCOUNT_ID}.r2.cloudflarestorage.com",
    aws_access_key_id=R2_ACCESS_KEY_ID,
    aws_secret_access_key=R2_SECRET_ACCESS_KEY,
    region_name="auto",
)

PREFIX = f"category-classifier/{VERSION}"
s3.upload_file("model.onnx", R2_BUCKET_NAME, f"{PREFIX}/model.onnx")
s3.upload_file("config.json", R2_BUCKET_NAME, f"{PREFIX}/config.json")
print(f"uploaded to s3://{R2_BUCKET_NAME}/{PREFIX}/")

## Promote to production

Deliberately not done from this notebook — Colab has no `gh`/repo access, so it can't create the public GitHub Release that makes the private `goodgorithm-models` bucket's contents actually downloadable (same reasoning as the sentiment model). After checking the eval numbers and spot-checks above, promote from a machine with an authenticated `gh` CLI:

```
cd training && uv run python r2_release.py --model category publish v3
```

See the `release-category-classifier` skill for the full checklist.